# Module 01 — Operational Metrics for Amazon Bedrock

## Overview

In this lab, you'll learn how to measure and analyze operational metrics for Large Language Models (LLMs) on Amazon Bedrock. By the end, you'll be able to measure cost, latency, and streaming performance across multiple models, publish custom metrics to CloudWatch, and make data-driven decisions about model selection.

## What You'll Learn

1. **Cost Metrics** — How token-based pricing works, how to calculate costs, and why tokenizer differences affect your bill
2. **Latency Metrics** — Server-side vs client-side latency, throughput (tokens/sec), and what drives response time
3. **TTFT vs TTLT** — How streaming (`converse_stream`) enables faster perceived responses, and how to measure inter-token latency
4. **CloudWatch Integration** — Publishing custom operational metrics with model-level dimensions for production monitoring

## How You'll Apply It

We'll use **email summarization** as our real-world use case — processing business emails through four different models and comparing their operational profiles side by side. This demonstrates how model choice directly impacts speed, cost, and output characteristics in a production-like scenario.

## Models Used

- **Amazon Nova 2 Lite** (`us.amazon.nova-2-lite-v1:0`) — Lightweight multimodal model optimized for high-volume, cost-effective tasks like document processing and content classification
- **Amazon Nova Pro** (`us.amazon.nova-pro-v1:0`) — Flagship multimodal model with advanced reasoning for complex document analysis and multi-step tasks
- **Anthropic Claude Haiku 4.5** (`us.anthropic.claude-haiku-4-5-20251001-v1:0`) — Fast, lightweight model with strong coding and agent performance at lower cost
- **Anthropic Claude Sonnet 5** (`us.anthropic.claude-sonnet-5`) — Anthropic's most capable Sonnet model, built for coding, agents, and professional work with a 1M token context window

> Model IDs are centralised in [`../model_config.py`](../model_config.py) — edit that file to swap models across all Foundational Evaluations notebooks.

## Prerequisites

- AWS account with Amazon Bedrock access enabled
- Access to all four models above in your region (enable via the Bedrock console → Model access)
- CloudWatch permissions for publishing custom metrics (`cloudwatch:PutMetricData`)
- Python 3.10+ with `boto3` and `pandas` installed

**Estimated time**: 30–45 minutes

## Setup and Dependencies

In [ ]:
%pip install -q -r ../requirements.txt

In [ ]:
import boto3
import json
import os
import time
import statistics
import pandas as pd
from typing import Dict
from IPython.core.display_functions import display
from IPython.display import Markdown

bedrock_client = boto3.client("bedrock-runtime")
cloudwatch = boto3.client('cloudwatch')

# Model IDs and On-Demand pricing are centralised in ../model_config.py
import sys
sys.path.append("..")
from model_config import MODEL_PRICING


def display_styled_table(df: pd.DataFrame, title: str, footnote: str):
    """Display a styled pandas DataFrame with a title and footnote."""
    styled = df.style.set_table_styles(
        [
            {"selector": "th", "props": "text-align: center;"},
            {"selector": "td:first-child", "props": "text-align: left;"},
        ]
    ).hide(axis="index")
    display(Markdown(title))
    display(styled)
    display(Markdown(footnote))


print("✓ Setup complete — Bedrock and CloudWatch clients initialized.")
print(f"✓ {len(MODEL_PRICING)} models configured: {', '.join(MODEL_PRICING.keys())}")

## 1. Cost Metrics

Understanding token usage is essential for cost optimization. Amazon Bedrock offers multiple pricing tiers to match different workload requirements:

- **On-Demand (Standard Tier)**: Pay-per-token with no upfront commitment. Provides consistent performance for everyday AI applications. Ideal for variable workloads and getting started.
- **Priority Tier**: Premium tier offering up to 25% better latency for time-sensitive applications. Requests receive preferential processing during high-demand periods.
- **Flex Tier**: Cost-effective option with lower pricing for non-urgent workloads that can tolerate increased latency. During high-demand periods, Flex requests receive lower priority relative to Standard. Ideal for evaluations, content summarization, and multi-step agentic workflows.
- **Batch Inference**: 50% lower pricing for asynchronous processing of large datasets submitted to S3.
- **Reserved Tier**: Guaranteed capacity with predictable performance for mission-critical applications requiring consistent throughput.

**This example uses On-Demand (Standard Tier) pricing**, where costs are calculated as: `(tokens / 1,000,000) × rate per 1M tokens`. Different models have different pricing structures based on input and output tokens. You select the tier per-request via the `serviceTier` parameter in the Converse API.

For complete pricing details, visit the [Amazon Bedrock pricing page](https://aws.amazon.com/bedrock/pricing/) or learn more about [service tiers](https://aws.amazon.com/bedrock/service-tiers/).

In [ ]:
def calculate_cost(model_id: str, input_tokens: int, output_tokens: int) -> Dict:
    """Calculate the cost of a model invocation based on token usage.
    Returns raw numeric values for downstream use and formatted strings for display."""
    if model_id not in MODEL_PRICING:
        return {"error": f"Pricing not available for {model_id}"}

    pricing = MODEL_PRICING[model_id]
    input_cost = (input_tokens / 1_000_000) * pricing["input"]
    output_cost = (output_tokens / 1_000_000) * pricing["output"]
    total_cost = input_cost + output_cost

    return {
        "input_cost": input_cost,
        "output_cost": output_cost,
        "total_cost": total_cost,
        "input_cost_usd": f"${input_cost:.6f}",
        "output_cost_usd": f"${output_cost:.6f}",
        "total_cost_usd": f"${total_cost:.6f}",
    }


print("✓ calculate_cost() defined — computes cost from token counts using MODEL_PRICING.")

### Comparing costs across models

Let's use `calculate_cost()` to see how the same workload (20K input tokens, 1.5K output tokens) costs across all four models. This highlights the price-performance tradeoff — cheaper models may be sufficient for simple tasks, while more expensive models offer stronger reasoning.

In [ ]:
# Calculate costs for sample token counts (20K input, 1.5K output)
SAMPLE_INPUT_TOKENS = 20_000
SAMPLE_OUTPUT_TOKENS = 1_500

rows = []
for model_id in MODEL_PRICING:
    cost = calculate_cost(model_id, SAMPLE_INPUT_TOKENS, SAMPLE_OUTPUT_TOKENS)
    pricing = MODEL_PRICING[model_id]
    rows.append(
        {
            "Model": model_id,
            "Price per 1M input tokens": f"${pricing['input']:.2f}",
            "Price per 1M output tokens": f"${pricing['output']:.2f}",
            "Input Cost (USD)": cost["input_cost_usd"],
            "Output Cost (USD)": cost["output_cost_usd"],
            "Total Cost (USD)": cost["total_cost_usd"],
        }
    )

df = pd.DataFrame(rows)

display_styled_table(
    df,
    "**Pricing Information**\n\n*Amazon Bedrock On-Demand pricing: you pay per token processed with no upfront commitment.*",
    "*Sample costs shown for {:,} input tokens and {:,} output tokens. Prices may change — verify current rates at the link above.*".format(SAMPLE_INPUT_TOKENS, SAMPLE_OUTPUT_TOKENS)
)

## 2. Latency Metrics

Latency measures how long it takes to get a complete response from the model. This is critical for user experience and varies significantly across models.

The Bedrock Converse API returns a `metrics.latencyMs` field in every response, which captures the **server-side processing time** (not including network round-trip). We also measure **client-side latency** using `time.time()` to capture the full picture including network overhead.

Factors that affect latency:
- **Model size**: Larger models (e.g., Claude Sonnet) are slower but often more capable
- **Prompt length**: More input tokens require more processing time
- **Output length**: More tokens to generate means longer total time
- **Region and service tier**: Priority tier offers up to 25% better latency

Below we compare all four models on the same prompt to see these differences in practice.

In [ ]:
def measure_latency(model_id: str, prompt: str, max_tokens: int = 100) -> Dict:
    """Measure server-side and client-side latency for a model invocation."""
    try:
        start_time = time.time()
        
        response = bedrock_client.converse(
            modelId=model_id,
            messages=[{"role": "user", "content": [{"text": prompt}]}],
            inferenceConfig={"maxTokens": max_tokens}
        )
        
        client_latency_ms = round((time.time() - start_time) * 1000, 2)
        server_latency_ms = response["metrics"]["latencyMs"]
        output_tokens = response["usage"]["outputTokens"]
        
        cost_info = calculate_cost(
            model_id,
            response["usage"]["inputTokens"],
            output_tokens
        )
        
        return {
            "model_id": model_id,
            "server_latency_ms": server_latency_ms,
            "client_latency_ms": client_latency_ms,
            "input_tokens": response["usage"]["inputTokens"],
            "output_tokens": output_tokens,
            "tokens_per_second": round(output_tokens / (server_latency_ms / 1000), 1) if server_latency_ms > 0 else 0,
            **cost_info,
            "error": False
        }
        
    except Exception as e:
        return {"model_id": model_id, "error": True, "error_message": str(e)}

print("✓ measure_latency() defined — measures server-side and client-side latency using the Converse API.")

### Latency comparison across all models

In [ ]:
# Compare latency across all models using the same prompt
LATENCY_PROMPT = "Explain quantum computing in simple terms."
LATENCY_MAX_TOKENS = 150

latency_rows = []
for model_id in MODEL_PRICING:
    print(f"Testing {model_id}...")
    result = measure_latency(model_id, LATENCY_PROMPT, max_tokens=LATENCY_MAX_TOKENS)
    
    if not result.get("error"):
        latency_rows.append({
            "Model": result["model_id"],
            "Server Latency (ms)": result["server_latency_ms"],
            "Client Latency (ms)": result["client_latency_ms"],
            "Input Tokens": result["input_tokens"],
            "Output Tokens": result["output_tokens"],
            "Tokens/sec": result["tokens_per_second"],
            "Cost (USD)": result["total_cost_usd"],
        })
    else:
        print(f"  Error: {result['error_message']}")
    
    time.sleep(0.5)  # Avoid throttling between calls

latency_df = pd.DataFrame(latency_rows)

display_styled_table(
    latency_df,
    "**Latency Comparison** — Same prompt, same max tokens across all models.",
    "*Server latency is from `response[\"metrics\"][\"latencyMs\"]` (server-side processing). "
    "Client latency includes network round-trip overhead.*"
)

### Analysis

Notice how the models differ across latency, throughput (tokens/sec), and cost for the same prompt. Smaller models like Nova 2 Lite tend to respond faster and cheaper, while larger models like Claude Sonnet produce richer output at higher latency and cost. Choosing the right model depends on whether your use case prioritizes speed, quality, or cost.

You may also notice that **input token counts differ across models** even though the same prompt was sent. This is because each model uses its own tokenizer — a "token" is not a universal unit. Think of it like measuring a hallway in "steps": a tall person and a short person cover the same distance in a different number of steps. The text (distance) didn't change, but the measurement unit did. Each model's vocabulary splits the same text into a different number of chunks.

**Why this matters for cost comparison:** You cannot compare "price per token" across models in isolation. A model charging $3/M tokens that tokenizes your prompt into 1,200 tokens may be cheaper than one charging $2/M tokens that tokenizes the same prompt into 2,000 tokens. The fair comparison is **cost per request** (the "Total Cost" column above), which accounts for tokenizer differences automatically.

Now let's dive deeper into streaming-specific metrics: TTFT and TTLT.

## 3. TTFT vs TTLT (Time to First Token vs Time to Last Token)

When using the **streaming API** (`converse_stream`), we can measure two additional timing metrics that aren't available with the synchronous `converse` call:

### Time to First Token (TTFT)
Measures how quickly a model begins generating its response after receiving a prompt. This is crucial for perceived responsiveness in user-facing applications.

- **Lower TTFT** creates the impression of a more responsive system
- Affected by model size, prompt complexity, and service tier

### Time to Last Token (TTLT)
Measures the total time from prompt submission to complete response delivery. This determines overall throughput.

- **Lower TTLT** enables processing more requests per unit time
- The gap between TTFT and TTLT is the **generation time** — how long the model spends producing tokens

### Inter-Token Latency
The average time between consecutive streamed tokens. This measures how smoothly tokens arrive — important for real-time display of streaming responses.

We also publish TTFT, TTLT, and cost as **custom CloudWatch metrics** so they can be monitored alongside Bedrock's built-in dashboards.

### Publishing custom metrics to CloudWatch

Before measuring streaming performance, we define a helper that publishes TTFT, TTLT, and cost as [custom CloudWatch metrics](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/publishingMetrics.html). This lets us visualize these metrics in dashboards alongside Bedrock's built-in metrics.

Each metric is tagged with a `Model` dimension so we can filter and compare per model.

In [ ]:
def put_custom_operational_cw_metrics(model_id: str, ttft_ms: float, ttlt_ms: float, total_cost: float):
    """Publish custom operational metrics (TTFT, TTLT, Cost) to CloudWatch."""
    metric_data = [
        {
            'MetricName': 'TimeToFirstToken',
            'Value': ttft_ms,
            'Unit': 'Milliseconds',
            'Dimensions': [{'Name': 'Model', 'Value': model_id}],
        },
        {
            'MetricName': 'TimeToLastToken',
            'Value': ttlt_ms,
            'Unit': 'Milliseconds',
            'Dimensions': [{'Name': 'Model', 'Value': model_id}],
        },
        {
            'MetricName': 'TotalCost',
            'Value': total_cost,
            'Dimensions': [{'Name': 'Model', 'Value': model_id}],
        },
    ]
    cloudwatch.put_metric_data(
        Namespace='llm_custom_operational_metrics',
        MetricData=metric_data,
    )


print("✓ put_custom_operational_cw_metrics() defined — publishes TTFT, TTLT, and Cost to CloudWatch.")

### Measuring streaming metrics

The function below uses `converse_stream()` to measure:
- **TTFT**: Time from request start to the first `contentBlockDelta` event
- **TTLT**: Time from request start to the last `contentBlockDelta` event
- **Generation time**: TTLT − TTFT (how long the model spent producing tokens)
- **Inter-token latency**: Average gap between consecutive streamed chunks

If a model doesn't support streaming, the function falls back to the synchronous `converse()` API — in that case TTFT is not measurable and is reported as `None`.

In [ ]:
def measure_streaming_metrics(model_id: str, prompt: str, max_tokens: int = 200) -> Dict:
    """Measure TTFT and TTLT using the streaming Converse API.
    Falls back to non-streaming converse API if streaming is not supported."""
    try:
        start_time = time.time()
        
        try:
            response_stream = bedrock_client.converse_stream(
                modelId=model_id,
                messages=[{"role": "user", "content": [{"text": prompt}]}],
                inferenceConfig={"maxTokens": max_tokens}
            )
            streaming_supported = True
        except Exception as stream_err:
            print(f"   Streaming not supported for {model_id}, falling back to non-streaming API...")
            streaming_supported = False
            response_obj = bedrock_client.converse(
                modelId=model_id,
                messages=[{"role": "user", "content": [{"text": prompt}]}],
                inferenceConfig={"maxTokens": max_tokens}
            )
        
        if not streaming_supported:
            # Non-streaming fallback: TTFT is not measurable, so we set it to None
            end_time = time.time()
            input_tokens = response_obj["usage"]["inputTokens"]
            output_tokens = response_obj["usage"]["outputTokens"]
            response_text = response_obj["output"]["message"]["content"][0]["text"]
            ttlt_ms = round((end_time - start_time) * 1000, 2)
            
            cost_info = calculate_cost(model_id, input_tokens, output_tokens)
            put_custom_operational_cw_metrics(model_id, ttlt_ms, ttlt_ms, cost_info["total_cost"])
            
            return {
                "model_id": model_id,
                "ttft_ms": None,
                "ttlt_ms": ttlt_ms,
                "generation_time_ms": None,
                "tokens_per_second": round(output_tokens / (ttlt_ms / 1000), 1) if ttlt_ms > 0 else 0,
                "avg_inter_token_latency_ms": None,
                "input_tokens": input_tokens,
                "output_tokens": output_tokens,
                "total_tokens_received": 0,
                "response_text": response_text,
                **cost_info,
                "error": False,
                "streaming": False
            }
        
        # Process the streaming response event by event
        first_token_time = None
        last_token_time = None
        token_timestamps = []
        input_tokens = 0
        output_tokens = 0
        response_text = ""
        
        for event in response_stream["stream"]:
            current_time = time.time()
            
            if 'contentBlockDelta' in event:
                if first_token_time is None:
                    first_token_time = current_time  # TTFT captured here
                
                last_token_time = current_time
                token_timestamps.append(current_time)
                
                if 'delta' in event['contentBlockDelta'] and 'text' in event['contentBlockDelta']['delta']:
                    response_text += event['contentBlockDelta']['delta']['text']
                
            elif 'metadata' in event:
                usage = event['metadata'].get('usage', {})
                input_tokens = usage.get('inputTokens', 0)
                output_tokens = usage.get('outputTokens', 0)
        
        end_time = last_token_time if last_token_time else time.time()
        
        # Calculate timing metrics
        ttft_ms = round((first_token_time - start_time) * 1000, 2) if first_token_time else None
        ttlt_ms = round((end_time - start_time) * 1000, 2)
        
        # Calculate inter-token latencies
        inter_token_latencies = []
        if len(token_timestamps) > 1:
            for i in range(1, len(token_timestamps)):
                inter_token_latencies.append(
                    (token_timestamps[i] - token_timestamps[i-1]) * 1000
                )
        
        cost_info = calculate_cost(model_id, input_tokens, output_tokens)
        put_custom_operational_cw_metrics(model_id, ttft_ms, ttlt_ms, cost_info["total_cost"])
        
        return {
            "model_id": model_id,
            "ttft_ms": ttft_ms,
            "ttlt_ms": ttlt_ms,
            "generation_time_ms": round(ttlt_ms - ttft_ms, 2) if ttft_ms else None,
            "tokens_per_second": round(output_tokens / (ttlt_ms / 1000), 1) if ttlt_ms > 0 else 0,
            "avg_inter_token_latency_ms": round(statistics.mean(inter_token_latencies), 2) if inter_token_latencies else None,
            "input_tokens": input_tokens,
            "output_tokens": output_tokens,
            "total_tokens_received": len(token_timestamps),
            "response_text": response_text,
            **cost_info,
            "error": False,
            "streaming": True
        }
        
    except Exception as e:
        return {"model_id": model_id, "error": True, "error_message": str(e)}


print("✓ measure_streaming_metrics() defined — measures TTFT, TTLT, and inter-token latency via converse_stream.")

### Streaming comparison across all models

Now we run all four models with the same prompt using `measure_streaming_metrics()` and compare their TTFT, TTLT, generation time, throughput, and inter-token latency side by side.

In [ ]:

# Compare streaming metrics across all models
STREAMING_PROMPT = "Write a short story about a robot learning to paint."
STREAMING_MAX_TOKENS = 300

streaming_rows = []
for model_id in MODEL_PRICING:
    print(f"Testing {model_id}...")
    result = measure_streaming_metrics(model_id, STREAMING_PROMPT, max_tokens=STREAMING_MAX_TOKENS)
    
    if not result.get("error"):
        streaming_rows.append({
            "Model": result["model_id"],
            "TTFT (ms)": result["ttft_ms"] or "N/A",
            "TTLT (ms)": result["ttlt_ms"],
            "Generation (ms)": result["generation_time_ms"] or "N/A",
            "Tokens/sec": result["tokens_per_second"],
            "Avg Inter-token (ms)": result["avg_inter_token_latency_ms"] or "N/A",
            "Output Tokens": result["output_tokens"],
            "Cost (USD)": result["total_cost_usd"],
            "Streaming": "✓" if result.get("streaming") else "✗",
        })
    else:
        print(f"  Error: {result['error_message']}")
    
    time.sleep(0.5)

streaming_df = pd.DataFrame(streaming_rows)

display_styled_table(
    streaming_df,
    "**Streaming Metrics Comparison** — TTFT, TTLT, and inter-token latency across all models.",
    "*TTFT = Time to First Token (via converse_stream). Generation = TTLT − TTFT. "
    "Inter-token latency is the average gap between consecutive streamed chunks.*"
)

### Visualizing operational metrics using CloudWatch Dashboard

Amazon CloudWatch has automatic dashboards for customers to quickly gain insights into the health and performance of their AWS services. An automatic dashboard for Amazon Bedrock is available with [Amazon Bedrock runtime metrics](https://docs.aws.amazon.com/bedrock/latest/userguide/monitoring.html#runtime-cloudwatch-metrics). To access Bedrock automatic dashboard from the AWS Management Console:

Select Dashboards from the CloudWatch console, and select the Automatic Dashboards tab. You’ll see an option for an Amazon Bedrock dashboard in the list of available dashboards. 

You can create a [custom CloudWatch Dashboard](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/create_dashboard.html) and add the Bedrock Automatic Dashboard to it as shown below: 

<div style="text-align:left">
    <img src="images/operational-metrics-cloudwatch-dashboard.png" width="100%"/>
</div>

## 4. Email Summarization Use Case

Now let's apply everything we've learned to a real-world scenario. We'll use all four models to summarize business emails and compare their operational metrics side by side — TTFT, TTLT, throughput, and cost.

This demonstrates how model choice directly impacts performance and cost in a production-like workload.

### Loading sample emails

We load email files from the `data/emails/` folder. Each file contains a business email that we'll summarize with all four models.

In [ ]:
import glob
from pathlib import Path

def load_emails_from_folder(folder_path="data/emails"):
    """Load email files from a folder and return structured email data."""
    sample_emails = []
    email_files = sorted(glob.glob(os.path.join(folder_path, "*.txt")))
    
    for i, file_path in enumerate(email_files, 1):
        try:
            with open(file_path, 'r', encoding='utf-8') as file:
                content = file.read().strip()
            
            filename = Path(file_path).stem
            
            if content.startswith("Subject:"):
                lines = content.split('\n')
                subject = lines[0].replace("Subject:", "").strip()
                email_content = '\n'.join(lines[1:]).strip()
            else:
                subject = filename.replace("_", " ").title()
                email_content = content
            
            sample_emails.append({"id": i, "subject": subject, "content": email_content})
        except Exception as e:
            print(f"Error reading {file_path}: {e}")
    
    return sample_emails


sample_emails = load_emails_from_folder()
print(f"✓ Loaded {len(sample_emails)} emails: {[e['subject'] for e in sample_emails]}")

### Summarization prompt and comparison function

We define a structured prompt that asks the model to extract key points, action items, deadlines, and impact from each email. The comparison function runs this prompt through all models using `measure_streaming_metrics()` and collects the results.

In [ ]:
summarization_prompt = """
You are an AI assistant that summarizes business emails for busy executives. 

Please analyze the following email and provide a concise summary that includes:

1. **Key Points**: Main topics and important information
2. **Action Items**: Specific tasks or decisions required
3. **Deadlines**: Any time-sensitive items
4. **People/Teams Involved**: Who needs to take action
5. **Impact**: Business impact or urgency level

Email to summarize:
{email_content}

Provide a clear, structured summary in 3-4 sentences followed by bullet points for action items.
"""


def run_email_summarization_comparison(email_data, models):
    """Compare models on email summarization task with performance metrics."""
    results = []
    
    print(f"\nTesting Email: '{email_data['subject']}'")
    print("-" * 50)
    
    for model_id in models:
        print(f"  Testing {model_id.split('.')[-1]}...", end=" ")
        
        prompt = summarization_prompt.format(
            email_content=f"Subject: {email_data['subject']}\n\n{email_data['content']}"
        )
        
        result = measure_streaming_metrics(
            model_id=model_id,
            prompt=prompt,
            max_tokens=400
        )
        
        if not result.get("error"):
            print(f"✓ TTFT={result['ttft_ms']}ms, TTLT={result['ttlt_ms']}ms, Cost={result['total_cost_usd']}")
            result['email_id'] = email_data['id']
            result['email_subject'] = email_data['subject']
            result['email_content'] = email_data['content']
            result['model_name'] = model_id.split('.')[-1]
            result['prompt_used'] = prompt
        else:
            print(f"✗ Error: {result['error_message']}")
        
        results.append(result)
        time.sleep(0.5)
    
    return results


print("✓ Summarization prompt and comparison function defined.")

### Running the comparison

Now we run all four models against each email and display the results in a summary table.

In [ ]:
all_results = []

for email in sample_emails:
    email_results = run_email_summarization_comparison(email, list(MODEL_PRICING.keys()))
    all_results.extend(email_results)

# Build summary table from all results
email_rows = []
for result in all_results:
    if not result.get("error"):
        email_rows.append({
            "Email": result['email_subject'],
            "Model": result['model_name'],
            "TTFT (ms)": result.get('ttft_ms') or "N/A",
            "TTLT (ms)": result['ttlt_ms'],
            "Tokens/sec": result['tokens_per_second'],
            "Output Tokens": result['output_tokens'],
            "Cost (USD)": result['total_cost_usd'],
        })

email_df = pd.DataFrame(email_rows)

display_styled_table(
    email_df,
    "**Email Summarization — Performance Comparison**",
    "*Each model summarized the same emails with max_tokens=400. "
    "Cost includes both input and output tokens.*"
)

### Analyzing Results

The table above shows how the same summarization task produces different operational profiles across models. You'll notice differences in speed (TTFT/TTLT), output verbosity (token count), and cost. However, we haven't yet inspected the quality of the summaries — which model gave the best responses for this use case? That's what we'll cover in the next module.

### Visualizing custom operational metrics

The `put_custom_operational_cw_metrics` function (called inside `measure_streaming_metrics`) [publishes custom metrics](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/publishingMetrics.html): TTFT, TTLT, and Total Cost to the `llm_custom_operational_metrics` [CloudWatch namespace](https://docs.aws.amazon.com/AmazonCloudWatch/latest/monitoring/cloudwatch_concepts.html#Namespace) for each model. This enables you to visualize these metrics in your observability dashboard as shown below:

<div style="text-align:left">
    <img src="images/Custom-operational-metrics-cloudwatch-dashboard.png" width="100%"/>
</div>

Similarly, you can create any additional custom metric you need for your application logic.

### Saving responses for quality evaluation

We save the model responses along with their performance metrics to a JSON file. This file will be used in **Module 02 (Quality Metrics)** to evaluate the quality of each model's summaries — allowing us to correlate operational performance with output quality.

In [ ]:
# Save responses for quality metrics evaluation in Module 02
enhanced_results = []

for result in all_results:
    if not result.get("error") and result.get("response_text"):
        enhanced_results.append({
            "email_id": result['email_id'],
            "email_subject": result['email_subject'],
            "email_content": result['email_content'],
            "model_id": result['model_id'],
            "model_name": result['model_name'],
            "agent_response": result['response_text'],
            "prompt_used": result['prompt_used'],
            "input_tokens": result['input_tokens'],
            "output_tokens": result['output_tokens'],
            "performance_metrics": {
                "ttft_ms": result.get('ttft_ms'),
                "ttlt_ms": result.get('ttlt_ms'),
                "tokens_per_second": result.get('tokens_per_second'),
                "total_cost_usd": result.get('total_cost_usd')
            }
        })

output_path = "email_responses.json"
with open(output_path, 'w') as f:
    json.dump(enhanced_results, f, indent=2)

print(f"✓ Saved {len(enhanced_results)} responses to {output_path}")
print(f"  Models: {set(r['model_name'] for r in enhanced_results)}")
print(f"  Emails: {set(r['email_subject'] for r in enhanced_results)}")

## 5. Key Takeaways and Best Practices

### Key Takeaways

Throughout this module, we explored the core operational metrics that every production LLM application needs to track. Here's what we learned:

- **Cost is driven by tokens, not requests.** Unlike traditional APIs where you pay per call, LLM pricing is based on the number of input and output tokens processed. The same prompt can produce different token counts across models because each model uses its own tokenizer. This means cost optimization starts with understanding your token usage patterns — not just your request volume.

- **Latency has two layers.** Server-side latency (`metrics.latencyMs`) tells you how long Bedrock took to process your request. Client-side latency adds network round-trip time on top. For production monitoring, track both — server-side for model performance, client-side for user experience.

- **Streaming changes the user experience.** With the synchronous `converse()` API, users wait for the entire response before seeing anything. With `converse_stream()`, the first token arrives much sooner (TTFT), creating a more responsive feel even if the total time (TTLT) is similar. Inter-token latency tells you how smoothly the stream flows.

- **Model selection is a tradeoff.** As we saw in the email summarization comparison, smaller models (Nova 2 Lite) are faster and cheaper but may produce less detailed output. Larger models (Claude Sonnet) deliver richer responses at higher cost and latency. There's no universally "best" model — the right choice depends on your use case requirements.

- **Observability is not optional.** Publishing custom CloudWatch metrics (TTFT, TTLT, cost) with model-level dimensions gives you the visibility to detect regressions, compare models in production, and set up alerts before issues impact users.

### Best Practices for Production

**Cost management:**
- Monitor token usage per model and per use case — set CloudWatch alarms for unexpected spikes
- Use the right [service tier](https://aws.amazon.com/bedrock/service-tiers/) for each workload: Standard for production, Flex for evaluations and batch work, Priority for latency-sensitive applications
- Consider [Batch Inference](https://docs.aws.amazon.com/bedrock/latest/userguide/batch-inference.html) (50% discount) for non-real-time processing like content generation pipelines
- Optimize prompts to reduce input token count — shorter, clearer instructions often produce better results at lower cost

**Latency optimization:**
- Use streaming (`converse_stream`) for user-facing applications to minimize perceived wait time
- Set appropriate `maxTokens` limits — don't request 4,000 tokens when 400 will do
- Implement timeouts and retry logic with exponential backoff for throttled requests
- Use [cross-region inference](https://docs.aws.amazon.com/bedrock/latest/userguide/cross-region-inference.html) for high availability and reduced throttling

**Monitoring and alerting:**
- Publish custom metrics to CloudWatch with meaningful dimensions (model, use case, environment)
- Use Bedrock's [built-in CloudWatch metrics](https://docs.aws.amazon.com/bedrock/latest/userguide/monitoring.html) alongside your custom metrics
- Set up alarms for latency P99, error rates, and cost thresholds
- Enable [model invocation logging](https://docs.aws.amazon.com/bedrock/latest/userguide/model-invocation-logging.html) for debugging and audit trails

**Scaling considerations:**
- Plan for rate limits — each model and tier has different RPM (requests per minute) and TPM (tokens per minute) quotas
- Request [service quota increases](https://docs.aws.amazon.com/bedrock/latest/userguide/quotas.html) proactively before scaling up
- Consider caching strategies for repeated or similar prompts to reduce both cost and latency

## Conclusion

In this module, you learned how to measure and analyze operational metrics for Amazon Bedrock — from cost calculation and latency measurement to streaming analysis and CloudWatch monitoring. You applied these metrics to a real-world email summarization use case and compared four models side by side.

However, operational metrics only tell you *how fast* and *how much* — they don't tell you *how good*. In the next module, we'll evaluate the quality of the model outputs to determine which model actually produces the best summaries for our use case.

### Next Steps
- **Module 02 — Quality Metrics**: Evaluate model output quality using programmatic testing and LLM-as-a-Judge
- **Module 03 — Understanding Failures**: Read traces, identify recurring failure patterns, and decide what to fix or evaluate
- **Module 04 — Agentic Metrics**: Evaluate agent performance, tool execution, and reliability
- **Optional Workload-Specific Evaluations**: IDP, Guardrails, Basic RAG, MultiModal RAG, Speech-to-Speech, Automated Reasoning, Tool Calling, Chatbot, Red Teaming, and Multiagent Shared Context
- **Optional Framework-Specific Evaluations**: PromptFoo, Strands, AgentCore, DSPy, MLflow, and DeepEval
